<a href="https://colab.research.google.com/github/Heng1222/MalSem-Decon/blob/main/method3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 安裝 BERTopic 與中文分詞工具
!pip install bertopic jieba sentence-transformers pandas numpy

import pandas as pd
import numpy as np
import jieba
import re
import zipfile
import xml.etree.ElementTree as ET
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

  Using cached bertopic-0.17.4-py3-none-any.whl.metadata (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.8 MB/s eta 0:00:00


In [5]:
def read_timeline_xlsx(path):
    # 直接解析 xlsx 的 XML，避免缺少 openpyxl 時讀檔失敗。
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rows = []

    def cell_text(cell):
        inline = cell.find("a:is", ns)
        value = cell.find("a:v", ns)
        return "".join(inline.itertext()) if inline is not None else (value.text if value is not None else "")

    with zipfile.ZipFile(path) as zf:
        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        sheet_rows = root.find("a:sheetData", ns)
        header_map = {}

        for row_idx, row in enumerate(sheet_rows):
            row_values = {}
            for cell in row:
                ref = cell.attrib.get("r", "")
                col = "".join(ch for ch in ref if ch.isalpha())
                row_values[col] = cell_text(cell)

            if row_idx == 0:
                header_map = row_values
                continue

            rows.append({header_map.get(col, col): value for col, value in row_values.items()})

    return pd.DataFrame(rows)

# --- [功能 A: 資料深度清洗] ---
def preprocess_fb_data(series_list):
    """
    解決 ValueError: Unsupported input type: float 的關鍵步驟。
    確保所有輸入皆為字串且非空。
    """
    df = pd.concat(series_list).dropna().astype(str).reset_index(drop=True)
    # 移除空白字串與過短的內容 (雜訊)
    df = df[df.str.strip().str.len() > 5].reset_index(drop=True)
    return df

# --- [功能 B: 中文分詞配置] ---
def tokenize_chinese(text):
    """
    針對 BERTopic 的 c-TF-IDF 進行中文分詞適配
    """
    words = jieba.lcut(re.sub(r'[^\u4e00-\u9fa5]', '', text))
    return " ".join([w for w in words if len(w) > 1])

# --- [功能 C: 初始化模型元件] ---
def get_bertopic_components():
    # 1. Embedding: 使用多國語言模型 (支援繁體中文)
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

    # 2. UMAP: 降維並保留局部拓撲結構
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')

    # 3. HDBSCAN: 自動發現聚類 (不預設 K 值)
    hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean',
                            cluster_selection_method='eom', prediction_data=True)

    # 4. Vectorizer: 處理中文分詞
    vectorizer_model = CountVectorizer(stop_words=None) # 停用詞可後續加入

    return embedding_model, umap_model, hdbscan_model, vectorizer_model

In [6]:
# 1. 載入資料 (請替換為你的 FB 資料)
# df_all = preprocess_fb_data([fb_groups_series, fb_posts_series])
# 模擬測試
posts = read_timeline_xlsx("time_line.xlsx")
text_columns = ["content", "share_content", "attachment_title", "attachment_description"]
posts[text_columns] = posts[text_columns].fillna("").astype(str)
posts["text"] = posts[text_columns].agg(" ".join, axis=1)
df_docs = posts["text"]
cleaned_docs = preprocess_fb_data([df_docs])

# 2. 中文預分詞 (BERTopic 處理中文的必要步驟)
processed_docs = cleaned_docs.apply(tokenize_chinese).tolist()

# 3. 建立並訓練模型
embed_m, umap_m, hdb_m, vec_m = get_bertopic_components()

topic_model = BERTopic(
    embedding_model=embed_m,
    umap_model=umap_m,
    hdbscan_model=hdb_m,
    vectorizer_model=vec_m,
    verbose=True
)

topics, probs = topic_model.fit_transform(processed_docs)

# 4. 提取結果：這就是你想要的「不好的概念」與「錨點詞」
topic_info = topic_model.get_topic_info()
print("\n>>> 自動發現的語意主題列表：")
print(topic_info.head(10))

# 儲存每個主題的前 20 個關鍵詞作為子空間錨點
topic_info.to_csv("malicious_concepts_taxonomy.csv", index=False)

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.608 seconds.
DEBUG:jieba:Loading model cost 0.608 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-04-25 13:56:11,720 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1065 [00:00<?, ?it/s]

2026-04-25 13:56:41,553 - BERTopic - Embedding - Completed ✓
2026-04-25 13:56:41,554 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-25 13:57:53,317 - BERTopic - Dimensionality - Completed ✓
2026-04-25 13:57:53,319 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-25 13:58:02,019 - BERTopic - Cluster - Completed ✓
2026-04-25 13:58:02,030 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-25 13:58:02,980 - BERTopic - Representation - Completed ✓



>>> 自動發現的語意主題列表：
   Topic  Count            Name                             Representation  \
0     -1  15645  -1_我們_可以_活動_自己   [我們, 可以, 活動, 自己, 真的, 分享, 大家, 時間, 一起, 一個]   
1      0   1591   0_好吃_料理_美食_牛肉   [好吃, 料理, 美食, 牛肉, 美味, 蛋糕, 晚餐, 口味, 口感, 奶油]   
2      1    740   1_生日_快樂_祝福_謝謝   [生日, 快樂, 祝福, 謝謝, 蛋糕, 感謝, 健康, 禮物, 開心, 今年]   
3      2    598   2_自己_人生_別人_生活   [自己, 人生, 別人, 生活, 事情, 不是, 不要, 情緒, 努力, 時候]   
4      3    546   3_謝謝_祝福_感謝_感恩   [謝謝, 祝福, 感謝, 感恩, 大家, 各位, 平安, 健康, 收到, 恭喜]   
5      4    535   4_抽獎_活動_留言_分享  [抽獎, 活動, 留言, 分享, 粉絲團, 貼文, 獎者, 抽出, 粉絲, 資格]   
6      5    523   5_影片_新聞_電影_授權   [影片, 新聞, 電影, 授權, 電視, 立新, 聞網, 好看, 最狂, 記者]   
7      6    479   6_畢業_老師_高中_學校   [畢業, 老師, 高中, 學校, 學生, 孩子, 同學, 大學, 典禮, 我們]   
8      7    476   7_台灣_台北_香港_高雄  [台灣, 台北, 香港, 高雄, 我們, 媒體, 他們, 新加坡, 真的, 一個]   
9      8    474   8_音樂_唱歌_鋼琴_首歌  [音樂, 唱歌, 鋼琴, 首歌, 演唱, 樂團, 小男孩, 歌曲, 歌詞, 聲音]   

                                 Representative_Docs  
0  [重要 還是 重要 其實 大家 心裡 有數 何必 口中 重要 人來 得罪 另一半 天色 希一...  
1            

In [7]:
# 視覺化主題分布
topic_model.visualize_topics()

# 針對特定句子進行解構分析 (投影到子空間)
test_sentence = "保證獲利，現在入金立享回饋"
test_processed = tokenize_chinese(test_sentence)
target_topic, prob = topic_model.transform([test_processed])

print(f"\n句子: '{test_sentence}'")
print(f"被歸類的主題 ID: {target_topic[0]}")
print(f"在該子空間的分量強度 (Confidence): {prob[0]:.4f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-25 13:58:13,924 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-25 13:58:32,302 - BERTopic - Dimensionality - Completed ✓
2026-04-25 13:58:32,303 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-25 13:58:32,306 - BERTopic - Cluster - Completed ✓



句子: '保證獲利，現在入金立享回饋'
被歸類的主題 ID: -1
在該子空間的分量強度 (Confidence): 0.0000
